# B2.3 · Vulnerability auditing — deterministic Semgrep, then the model pass

**Function B — Application Security with an AI SDLC → The AI SDLC: an Agentic AppSec Pipeline, Before and After Deploy**  ·  *AI for Security*

Builds on **[B2.2 · Threat modelling from what the estate already knows](https://spbreed.github.io/cyber-commons/lessons/B2.2.html)**.

| | |
|---|---|
| Tools used | OpenGrep, Semgrep OSS, CodeQL, GLM-4.6, Kimi K2, Claude Sonnet 5 |

## What this lesson is

**What it covers.** Score grep, taint rules and model review against the same corpus, then combine them behind a confidence gate.

**Why a security engineer needs it.** Pattern matching floods the queue; the false-positive rate is what actually changed. The control it builds is: stage 7: deterministic rules for what rules do well, model reasoning for what rules cannot express.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Semgrep's precision on this file is 1.00 at every ruleset width. Its recall goes from 0.17 to 0.67 depending on a config line nobody reads, and both scans exit zero. The number that decides whether a scan meant anything is the one almost nobody computes.

> **At CyberTravels.** The IDOR that exposed card details by booking ID (R8) is exactly the class each generation of SAST handles differently — and the class the third generation will also confidently invent.

## 2 · The framework

```
   booking.py — 6 defects in the key, 5 of them a pattern

   DETERMINISTIC — semgrep 1.176.0        precision   recall
     p/python + p/secrets      1 finding     1.00       0.17
     seven registry packs      4 findings    1.00       0.67
     custom taint rule         2 findings    1.00       0.33
                                            ^^^^^      ^^^^^
                                     never wrong    a config line

   missed by every width:
     line 22  hardcoded key   -> COVERAGE GAP   . write the rule
     line  7  no authz check  -> NOT EXPRESSIBLE. no syntax to match

   PROBABILISTIC — the model pass, on line 7 only
     finds it  0.82   -> HYPOTHESIS, not a finding
     invents one 0.71 -> REJECTED: quoted a line not in the file

   one can gate a merge and misses things. the other finds what has
   no syntax and invents things. the gap between 0.82 and 0.71 is
   not a threshold - it is one sample
```

**Stage 7 — Vulnerability auditing.** The stage people mean when they say
"SAST", and the one where the two halves of this pipeline are easiest to
confuse with each other.

**The deterministic half is a real scanner with real rules.** Semgrep, CodeQL,
OpenGrep. Parse the code to a graph, ask a rule a question about it, and get
the same answer every time. That repeatability is what lets you gate a merge on
it — a probabilistic check cannot block a build, because the same commit would
pass on Tuesday and fail on Wednesday.

Its limit is not accuracy. On the file below Semgrep's precision is **1.00 at
every ruleset width**; it does not report bugs that are not there. Its limit is
that a rule only finds the pattern somebody wrote, so its **recall is a
configuration decision** — and one that is invisible, because a narrow scan and
a wide scan both exit `0`.

**The probabilistic half reads the code and reasons.** No rule needs to exist
first, which is exactly its value, and it is the only thing that reaches a
defect that is the **absence** of a call. It also invents defects, confidently,
with a similar-looking confidence number attached.

So the two are not competing generations where the newer one wins. They answer
different questions and fail in opposite directions, and this lesson runs each
as its own skill:

| | deterministic — Semgrep | probabilistic — the model pass |
|---|---|---|
| same answer twice | yes | no |
| can gate a merge | **yes** | no |
| finds what no rule expresses | no | **yes** |
| typical failure | missed it entirely | reported it and it was not there |
| output is a | **finding** | **hypothesis** |

That last row is the load-bearing one. Everything the model says enters the
pipeline as a hypothesis, and stages 8–12 are what turn one into a finding.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · The file, and the key written before anything ran

This is a real pull request from CyberTravels' Coding Agent, in
[`labs/tools/semgrep-sast/booking.py`](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/tools/semgrep-sast).
Six defects, enumerated by hand **before** any scanner saw it — because a key
written after the scan is a description of the scan.

```python
def find_booking(reference):                                    # 7  CWE-862
    cur.execute("SELECT * FROM bookings WHERE reference LIKE '%" + reference + "%'")   # 9  CWE-89
def render_itinerary(template, booking):
    return eval(template, {"booking": booking})                 # 14 CWE-95
def sync_vendor(vendor_host):
    subprocess.run("curl -s https://" + vendor_host + "/manifest", shell=True)  # 17 CWE-78
def notify(url, payload):
    return requests.post(url, json=payload, verify=False)       # 20 CWE-295
API_KEY = "sk-live-4f9a2b1c8e7d6a5b3c2d1e0f9a8b7c6d"            # 22 CWE-798
```

Five of the six are the *presence* of a pattern. The sixth, on line 7, is the
**absence** of one: `find_booking` returns a booking to whoever asks, and the
Workflow Agent calls it holding `payments.refund`. Hold on to that line — it is
the whole reason this lesson has two skills in it.

## 4 · Real Semgrep, at three widths, with the rule as a file

Not a forty-line taint engine written to fit in a lesson. Semgrep **1.176.0**,
against that file, three configurations —
[`run.sh`](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/tools/semgrep-sast)
reproduces all three and the raw JSON is committed beside the skill.

The third is a custom taint rule, which is what a real one looks like:

```yaml
rules:
  - id: cybertravels-sql-concat
    languages: [python]
    severity: ERROR
    message: >-
      Caller-controlled input is concatenated into a SQL string. Use a
      parameterised query.
    mode: taint
    pattern-sources:
      - pattern: |
          def $F(..., $X, ...):
            ...
    pattern-sinks:
      - pattern: $CUR.execute(...)
```

`mode: taint` is the whole difference between generations of scanner: not a
better pattern, a different question — *does caller-controlled input reach this
sink?*

## 5 · The deterministic half, as a skill

The skill scores each of the three real runs against the six-defect key and reports precision and recall **separately**, because merging them into one "accuracy" number hides the only one that moves. Then it partitions what every width missed into the two classes that matter: a defect a rule *could* match and nobody wrote (write the rule) and a defect no rule can express (the next skill).

### The skill — [`skills/appsec/sast-semgrep-deterministic/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/sast-semgrep-deterministic/SKILL.md)

```yaml
name: sast-semgrep-deterministic
description: >-
  Score a real Semgrep run against a ground-truth key and report recall per
  ruleset width, not just the finding count. Use when choosing or reviewing a
  SAST configuration, when a scan comes back clean, or when asked how much of
  the codebase a scanner is actually looking at.
allowed-tools: Read, Grep, Glob, Bash
```

# What did Semgrep look for, and what did it never look for?

Deterministic SAST parses code to a graph and asks a rule a question about it.
Given the same file and the same rules it returns the same findings, every
time — which is what makes it the half of the audit you can gate a merge on.

The mistake is reading its output as a measure of the file. It is a measure of
the **rules that were enabled**. A scan with one pack and a scan with seven
disagree by a factor of four on the same file and both exit `0`, so the number
that matters is never the finding count — it is **recall against a key**, and
almost nobody computes it because it requires knowing the answer in advance.

## When to use this

Before trusting any clean scan, and whenever a SAST configuration is chosen,
widened or inherited. Also the honest way to compare two scanners: run both
against a file whose defects you have enumerated by hand, and compare recall
rather than volume.

## Procedure

**1 — Enumerate the defects by hand first.** One line per defect, with its CWE
and whether it is expressible as a pattern at all. Do this before running
anything; a key written after the scan is a description of the scan.

**2 — Run the scanner at every width you are considering.** The default pack,
the widest set your team would plausibly enable, and any custom rules you
wrote. Keep the raw JSON — the finding's rule id is what tells you *which*
rule fired, and that is what you tune.

**3 — Score each width against the key.** Precision and recall separately.
Deterministic SAST usually has excellent precision and the recall is the
number under discussion; reporting them merged as "accuracy" hides exactly
that.

**4 — Partition what was missed into two classes.** A defect a rule *could*
match but none did is a **coverage gap** — someone writes the rule. A defect
that is the **absence** of a call, or depends on the caller's authority, is
not expressible as a pattern and no amount of rule-writing reaches it. Only
the second class justifies a model pass.

**5 — Report the width, not just the result.** Every finding count is
meaningless without the config that produced it. A report that says "Semgrep:
1 finding" and not "p/python + p/secrets" has withheld the part that decides
whether the scan meant anything.

## Output contract

```json
{
  "key": {"defects": 0, "pattern_expressible": 0},
  "runs": [{"config": "str", "findings": 0, "true_positives": 0,
            "precision": 0.0, "recall": 0.0}],
  "missed_by_all": [{"line": 0, "cwe": "str", "class": "coverage-gap|not-expressible"}],
  "widest_recall": 0.0
}
```

`class: not-expressible` is the only honest argument for adding a
probabilistic reviewer. If every miss is a coverage gap, the fix is a rule and
it is cheaper.

## Failure modes

- **Reading a clean scan as a clean file.** It means no enabled rule matched.
- **Comparing scanners on finding count.** The noisier one wins, which is
  backwards.
- **Writing the key after the run.** It converts recall into 1.00 by
  construction.
- **Enabling every pack to fix recall.** Recall rises and so does triage cost;
  the widest width here still misses a third of the key, so width alone was
  never going to be the answer.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/sast-semgrep-deterministic/scripts/sast_semgrep_deterministic.py
SCRIPT = "skills/appsec/sast-semgrep-deterministic/scripts/sast_semgrep_deterministic.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## 6 · Which of the two misses justifies a model

One of them does not. Line 22 is a hardcoded key — lexical, and `p/secrets` was
enabled and did not fire only because the string matches no known provider's
format. That is a rule somebody writes this afternoon, and reaching for a model
to find it is buying a language model to do a regex's job.

Line 7 is different in kind. There is no syntax for "this function should have
called `require_owner` and did not", and there is no width of ruleset that
reaches it — the defect only exists relative to the authority the caller holds,
which is in a different file. That is the boundary, and it is narrow. Cross it
deliberately and you have a reason to spend the model pass; cross it because
the deterministic scan felt disappointing and you have bought noise.

## 7 · The probabilistic half, as a skill

The same adapter every model-facing skill in this commons uses: offline a labelled replay, and against a served open-weight model the identical code. It is asked one question with a checkable answer, over the smallest slice in which the defect is decidable — the function, its signature, and the authority its caller holds.

It reviews two functions. One has the defect. The other is already parameterised and already authorised, and it is in there because a review pass that is never wrong has not been tested.

### The skill — [`skills/appsec/sast-model-pass/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/sast-model-pass/SKILL.md)

```yaml
name: sast-model-pass
description: >-
  Run a language model over a slice that deterministic rules structurally cannot
  reach, and record what it says as a hypothesis rather than a finding. Use
  after a rule-based scan has been scored, when a defect is the absence of a
  call rather than the presence of a pattern, or when deciding whether an
  unverified model confidence score may gate a pipeline.
allowed-tools: Read, Grep, Glob
```

# The pass that finds what no rule can express — and the price of it

A rule matches syntax that is present. Some defects are the **absence** of
something: no authorisation check, no ownership comparison, no expiry. There
is nothing to match, so no ruleset finds them at any width, and that is the
only defensible reason to put a model in an audit pipeline.

The price is that a model also reports defects that are not there, in the same
tone, with a similar confidence number. So the output of this pass is not a
finding list. It is a **hypothesis list**, and something else promotes a
hypothesis to a finding.

## When to use this

After the deterministic scan is scored, on the files where the key says a
defect exists and the rules were silent — never as a first pass over the whole
repository. A model asked to review everything reviews nothing carefully, and
at four million lines the pass costs more than the finding is worth.

## Procedure

**1 — Take only the slice where the defect is decidable.** The function, its
signature, and the authority its caller holds. The signature is the part
people drop and it is the part that decides the answer.

**2 — Ask a question with a checkable answer.** "Review this for security"
returns an essay. "Does this function verify that the caller owns the record
it returns — yes or no, and quote the line that does it" returns something you
can check without a second opinion.

**3 — Verify the quote against the file.** If the model quotes a line, the
line must exist. If it names a symbol, the symbol must be in the file. This
costs nothing and kills the largest class of model error before a human sees
it.

**4 — Record every survivor as `hypothesis`, never `finding`.** Carry the
model, the prompt slice and the confidence with it, so the promotion decision
downstream can be argued with rather than inherited.

**5 — Do not gate on the confidence number.** It is not calibrated and it is
not stable across runs. Run the same slice ten times before you let any
threshold into a pipeline; the variance is usually wider than the gap between
your accept and reject bands.

## Output contract

```json
{
  "backend": {"kind": "replay|open-weight", "model": "str"},
  "hypotheses": [{"unit": "str", "cwe": "str", "confidence": 0.0,
                  "quote_verified": true, "status": "hypothesis"}],
  "rejected": [{"unit": "str", "why": "quote-not-in-file|symbol-absent"}],
  "promoted": 0
}
```

`promoted` is 0 here on purpose. This skill does not promote anything —
deduplication, contextual verification, reachability and dynamic validation
do, and a pipeline that lets the audit stage promote its own output has no
independent step in it at all.

## Failure modes

- **Reporting model output as findings.** They enter a queue people trust and
  leave it as lost credibility.
- **Trusting the confidence.** Uncalibrated, unstable, and the first thing an
  eager pipeline gates on.
- **Sending the whole file.** The defect gets diluted and the answer degrades;
  the slice that makes it decidable is smaller than the file.
- **Dropping the signature from the slice.** The identical body is a critical
  defect in a handler and irrelevant in a migration script.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/sast-model-pass/scripts/sast_model_pass.py
SCRIPT = "skills/appsec/sast-model-pass/scripts/sast_model_pass.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## 8 · Where it breaks — gating on the confidence number

The model returned 0.82 on the real defect and 0.71 on the invented one. It is
tempting to read a threshold into that gap, and every pipeline that does it
ships one.

Two reasons not to. The number is **uncalibrated** — 0.82 does not mean the
claim is right 82% of the time, it means nothing in particular. And it is
**unstable**: the same slice, the same model, ten runs, and the confidence
moves further than the distance between your accept and reject bands. Sort a
human's queue with it if you like. Do not let it decide anything on its own.

What killed the false positive instead cost nothing and required no judgement:
the model quoted a line that is not in the file. That check generalises, and
B2.4 is where it becomes a stage.

## 9 · An agent drives both, because you cannot afford both everywhere

Semgrep is cheap enough to run over the whole repository. The model pass is
not — at four million lines it costs more than the finding is worth, and a
model asked to review everything reviews nothing carefully.

So neither half is the interesting part. **The allocation is**, and the policy
is three rules:

1. run the deterministic scanner everywhere, at the widest ruleset that is not
   noisy, because it is nearly free and it can gate the merge;
2. spend the model pass only where stage 1 says risk lives **and** the rules
   were silent — silence in a high-risk zone is the signal, not the noise;
3. mark everything the model says as a hypothesis, never a finding.

Get rule 2 wrong in the cheap direction and the authorisation defect on line 7
is never reviewed by anything, because the deterministic scan was green and
nobody was surprised by that.

## What you just proved

Semgrep's precision is 1.00 at all three widths and its recall is not: 0.17 on the default Python pack, 0.67 across seven registry packs, 0.33 on the custom taint rule — same file, same engine, and both scans exit 0. Two defects survive every width, and the skill separates them: the hardcoded key is a coverage gap somebody fixes by writing a rule, and the missing authorisation check on line 7 is not expressible as a pattern at any width. The model pass then finds exactly that one, at 0.82 confidence, recorded as a hypothesis and not a finding — and its claim about the already-authorised control function is rejected for nothing more than quoting a line that is not in the file. Zero hypotheses are promoted, because the audit stage does not promote its own output.

## Your turn

Two things, and the second is the one people skip. Run Semgrep against one of your own repositories at your current ruleset and at seven packs, and count the difference — whatever that number is, it has been the number all year. Then point the model pass at a real GLM-4.6 or Kimi K2 through Ollama and run one slice ten times. The variance in what it reports, and in its confidence, is what decides whether you can gate on confidence at all, and you cannot learn it from one run.

---

**Next → [B2.4 · Deduplication and contextual verification](https://spbreed.github.io/cyber-commons/lessons/B2.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*